In [ ]:
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from shapely.ops import linemerge

from geofeatureviz import river_preprocessor
from geofeatureviz.data import OverpassAPIHandler, path_settings

# Rivers data creation
This notebook explains how the data collection for the German rivers that I want to learn in my Anki deck worked, since it was less straight-forward than expected.

## Relations

I already created a file where all rivers that I want to have in my Anki deck are saved. The geographical information will be gathered from OSM using the Overpass API from the OSM-type and OSM-IDs of the rivers in this file.

In [ ]:
osm_id_df = pd.read_csv(path_settings.german_rivers_osm_id_path, index_col=0)

river_ids_str = ";".join(
    [f"{row.osm_type}({row.osm_id})" for row in osm_id_df.itertuples()]
)
river_ids_query = f"({river_ids_str};)"
api_handler_river_id = OverpassAPIHandler(
    file_path=path_settings.data_raw_dir / "osm_rivers" / "german_rivers_body_id.json"
)
api_handler_river_id.create_query(river_ids_query, output="body")
_ = api_handler_river_id.get()
river_id_df = api_handler_river_id.parse_json().set_index("id")

## River members
Here, we didn't ask for the geometries in our response yet. A *relation* like a river consists of several *members*, which together form the river. In general, river relations consist of several small parts, each having their own geometry. However, my preferred representation would be a single line geometry for each river. Even though this loses some information (e.g., a river delta consisting of several streams), but it simplifies the geometry and the further work with the river geometry.

To achieve this, we have to do a lot of preprocessing. To gain all information about each member of each relation, we need to do a request for each member (and not a single one only for the relation). With our current request, we can find out the members of a relation and fetch information about the members, where we include the geometries.

In [ ]:
member_gdfs = []

for i in range(100):
    try:
        for relation in river_id_df.itertuples(name="Relation"):
            members = [m for m in relation.members if m["type"] == "way"]

            member_refs_str = ";".join([f"way({m['ref']})" for m in members])
            query = f"({member_refs_str};)"

            # make api request / load old request
            api_handler_members = OverpassAPIHandler(
                file_path=path_settings.data_raw_dir
                / "osm_rivers"
                / "river_members"
                / f"{relation.name}_{relation.Index}_members.json",
            )
            api_handler_members.create_query(query, output="geom", timeout=300)
            # parse to GeoDataFrame
            _ = api_handler_members.get()
            member_gdf = api_handler_members.parse_json()
            # set role of members in relation
            member_roles_dict = {
                m["ref"]: m["role"] for m in members if m["role"] != ""
            }
            member_roles = [member_roles_dict.get(ref, "") for ref in member_gdf["id"]]
            member_gdf["role"] = member_roles
            # add some info about the relation to the member_gdf for easier access later
            member_gdf["relation_id"] = relation.Index
            member_gdf["relation_name"] = relation.name
            member_gdf = member_gdf.set_index("id")

            member_gdfs.append(member_gdf)
        break
    except Exception as e:
        print(f"Attempt {i + 1} of 100 failed with:\n{e}\nRetrying in 2 seconds...")
        time.sleep(2)

## Preprocessing
Here began the tedious part: creating a single line string from all members of a river relation.

The most obvious thing is to do the *shapely*-operation `linemerge`. However, for a lot of rivers, this returned a MultiLineString because of different reasons:
- side streams
- branches
- unclear sources

and many more. It is possible to clean up the member data frames (e.g., remove side streams), which already helped a lot. After that, only manual inspection helped to get rid of members or nodes that I didn't want in my data set. This manual inspection consisted of creating an inspection figure, where I could find the member ID's where merging the lines failed. on openstreetmap.org, I took a look at these members. In most cases, it was sufficient to just exclude this members, sometimes it was necessary to remove single nodes from the members. I saved the results of the manual inspection in a config file.

Here, all this is already applied under the hood in *river_preprocessing.py*, so in theory, all rivers should consist of a single LineString afterward.

In [ ]:
river_id_to_geom = {}
for member_gdf in member_gdfs:
    relation_id = member_gdf["relation_id"].iloc[0]
    relation_name = member_gdf["relation_name"].iloc[0]

    member_gdf = river_preprocessor.prep_member_gdf(member_gdf)
    geom = river_preprocessor.member_gdf_to_linestring(member_gdf)

    if geom.geom_type == "MultiLineString":
        # this was used for the manual cleaning; the resulting geometries should never
        # be MultLineStrings, since it was the goal of the cleaning to prevent this
        fig, ax = river_preprocessor.inspect_river_geom(geom, member_gdf)

        fig.savefig(f"{relation_id}_{relation_name}_inspect_delete.svg")
    else:
        river_id_to_geom[relation_id] = geom

river_gdf = gpd.GeoDataFrame(
    river_id_df, geometry=list(river_id_df.index.map(river_id_to_geom)), crs="EPSG:4326"
)

# save
file_path = path_settings.data_processed_dir / "osm_10m_rivers_germany.geojson"
if not file_path.exists():
    river_gdf.to_file(file_path, driver="GeoJson")

In addition to preprocessing, we can now conveniently simplify our river network.

To see the results of our preprocessing compared to the unprocessed data, we plot both into a single map here.

In [ ]:
# simplify
simplified_gdf = river_gdf.copy()
simplified_gdf.geometry = simplified_gdf.geometry.simplify(
    tolerance=0.03, preserve_topology=True
)

# get unprocessed data
river_id_to_geom_unprocessed = {
    gdf["relation_id"].iloc[0]: linemerge(gdf.geometry.to_list()) for gdf in member_gdfs
}
river_gdf_unprocessed = gpd.GeoDataFrame(
    river_id_df,
    geometry=list(river_id_df.index.map(river_id_to_geom_unprocessed)),
    crs="EPSG:4326",
)

# plot the difference
fig, ax = plt.subplots()
plt.rcParams["font.family"] = "Ubuntu"
plt.rcParams["svg.fonttype"] = "none"
colors = plt.get_cmap("Dark2").colors
linewidth = 2
ax.set_axis_off()

river_gdf_unprocessed.plot(
    ax=ax, color=colors[1], linewidth=linewidth, label="Original"
)
simplified_gdf.plot(ax=ax, color=colors[0], linewidth=linewidth, label="Aufbereitet")

ax.legend(fontsize=12)
plt.show()
# fig.savefig("preprocessing_comparison.svg", bbox_inches="tight", pad_inches=0)

We can see that not a lot is missing, mostly small side streams. This big things are:
- side stream of the Rhine
- delta of the Danube

For both, this is intended and follows my idea of having just a single line string for each river.

# Disk Space Comparison
When we compare the space that the unprocessed and the simplified GeoDataFrame would take up on disk if saved as GeoJSON, we can see that the simplified takes more than 10 times less space.

In [ ]:
def get_geojson_size(gdf: gpd.GeoDataFrame) -> int:
    """Get the file size of a GeoDataFrame when converted to GeoJSON format."""
    geojson_str = gdf.to_json()
    return len(geojson_str.encode("utf-8"))


size_unprocessed = get_geojson_size(river_gdf_unprocessed)
size_simplified = get_geojson_size(simplified_gdf)
size_comp_str = f"""Unprocessed GeoDataFrame as GeoJSON: \t{size_unprocessed:7} bytes
Simplified GeoDataFrame as GeoJSON: \t{size_simplified:7} bytes
-> ~{size_unprocessed / size_simplified:.2f} times smaller"""
print(size_comp_str)